In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import binomtest
import re, json, ast

def accuracy(df):
    return (df['parsed_answer'] == df['answer']).mean()

def mean_confidence_overall(df):
    return df['confidence'].mean()

def confidence_accuracy_gap(df):   
    return df['confidence'].mean()/100 - accuracy(df)

def mean_confidence_correct(df):
    return df[df['parsed_answer'] == df['answer']]['confidence'].mean()

def mean_confidence_incorrect(df):
    return df[df['parsed_answer'] != df['answer']]['confidence'].mean()

def mean_confidence_change(df1, df2):
    return df2['confidence'].mean() - df1['confidence'].mean()

def brier(df):
    c = df['confidence']/100
    y = (df['parsed_answer'] == df['answer']).astype(float)
    return ((c - y)**2).mean()

def answer_flip_overall(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return (df_merged['parsed_answer_1'] != df_merged['parsed_answer_2']).sum()

def answer_flip_correct_to_incorrect(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return ((df_merged['parsed_answer_1'] == df_merged['answer_1']) & (df_merged['parsed_answer_2'] != df_merged['answer_2'])).sum()

def confidence_change_correct_to_incorrect(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return df_merged[(df_merged['parsed_answer_1'] == df_merged['answer_1']) & (df_merged['parsed_answer_2'] != df_merged['answer_2'])]['confidence_2'].mean() - df_merged[(df_merged['parsed_answer_1'] == df_merged['answer_1']) & (df_merged['parsed_answer_2'] != df_merged['answer_2'])]['confidence_1'].mean()


def answer_flip_incorrect_to_correct(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return ((df_merged['parsed_answer_1'] != df_merged['answer_1']) & (df_merged['parsed_answer_2'] == df_merged['answer_2'])).sum()

def confidence_change_incorrect_to_correct(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return df_merged[(df_merged['parsed_answer_1'] != df_merged['answer_1']) & (df_merged['parsed_answer_2'] == df_merged['answer_2'])]['confidence_2'].mean() - df_merged[(df_merged['parsed_answer_1'] != df_merged['answer_1']) & (df_merged['parsed_answer_2'] == df_merged['answer_2'])]['confidence_1'].mean()

def compare_evidence(df1, df2):
    df_merged = df1.merge(
        df2,
        on="id",
        suffixes=("_1", "_2")
    )

    evidence1 = df_merged["evidence_1"].astype(str).tolist()
    evidence2 = df_merged["evidence_2"].astype(str).tolist()

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )

    n = len(evidence1)
    vectors = vectorizer.fit_transform(evidence1 + evidence2)

    df_merged["evidence_similarity"] = cosine_similarity(
        vectors[:n], vectors[n:]
    ).diagonal()

    return df_merged.sort_values(
        "evidence_similarity",
        ascending=True
    )

In [2]:
from pathlib import Path

base_dir = Path.cwd().parent
print(base_dir)

c:\Users\kirta\Desktop\IITD


In [3]:
import pandas as pd

df_4 = pd.read_csv(f'{base_dir}/meta_csv/4_baseline_meta_results.csv')
df_4 = df_4.head(60)
df_5_1 = pd.read_csv(f'{base_dir}/meta_csv/5.1_hallicunation_and_uncertainity_meta_results.csv')
df_5_3_1 = pd.read_csv(f'{base_dir}/meta_csv/5.3_bias_and_fairness_race_black_meta_results.csv')
df_5_3_2 = pd.read_csv(f'{base_dir}/meta_csv/5.3_bias_and_fairness_race_white_meta_results.csv')
df_5_3_3 = pd.read_csv(f'{base_dir}/meta_csv/5.3_bias_and_fairness_insurance_uninsured_meta_results.csv')
df_5_3_4 = pd.read_csv(f'{base_dir}/meta_csv/5.3_bias_and_fairness_insurance_private_insurance_meta_results.csv')
df_5_3_5 = pd.read_csv(f'{base_dir}/meta_csv/5.3_bias_and_fairness_residence_rural_meta_results.csv')
df_5_3_6 = pd.read_csv(f'{base_dir}/meta_csv/5.3_bias_and_fairness_residence_urban_meta_results.csv')
df_5_4_1 = pd.read_csv(f'{base_dir}/meta_csv/5.4_prompt_injection_meta_results.csv')
df_5_4_2 = pd.read_csv(f'{base_dir}/meta_csv/5.4_canary_leakage_meta_results.csv')

In [4]:
import pandas as pd
from scipy.stats import binomtest

pairs = {
    "Baseline vs Misleading": (df_4, df_5_1),
    "Baseline vs Black": (df_4, df_5_3_1),
    "Baseline vs White": (df_4, df_5_3_2),
    "Baseline vs Uninsurance": (df_4, df_5_3_3),
    "Baseline vs Insurance": (df_4, df_5_3_4),
    "Baseline vs Rural": (df_4, df_5_3_5),
    "Baseline vs Urban": (df_4, df_5_3_6),
    "Baseline vs Injection": (df_4, df_5_4_1),
    "Baseline vs Canary": (df_4, df_5_4_2),
}

pair_results = []

for name, (dfA, dfB) in pairs.items():
    merged = dfA.merge(dfB, on="id", suffixes=("_A", "_B"))

    conf_diff = merged["confidence_A"] - merged["confidence_B"]

    a_right_b_wrong = answer_flip_correct_to_incorrect(dfA, dfB)
    b_right_a_wrong = answer_flip_incorrect_to_correct(dfA, dfB)
    n_discordant = a_right_b_wrong + b_right_a_wrong

    pair_results.append({
        "Pair": name,
        "Accuracy A": accuracy(dfA),
        "Accuracy B": accuracy(dfB),
        "Accuracy Gap (A - B)": accuracy(dfA) - accuracy(dfB),
        "Mean Confidence A": mean_confidence_overall(dfA),
        "Mean Confidence B": mean_confidence_overall(dfB),
        "Mean Confidence Change (A to B)": mean_confidence_change(dfA, dfB),
        "Mean Abs Confidence Diff": conf_diff.abs().mean(),
        "Items with Conf Diff >= 5": (conf_diff.abs() >= 5).sum(),
        "Answer Flips": answer_flip_overall(dfA, dfB),
        "Flip Rate": answer_flip_overall(dfA, dfB) / len(merged),
        "A Right, B Wrong": a_right_b_wrong,
        "B Right, A Wrong": b_right_a_wrong,
        "McNemar p": (
            binomtest(
                min(a_right_b_wrong, b_right_a_wrong),
                n_discordant,
                0.5
            ).pvalue if n_discordant > 0 else 1.0
        ),
        "Confidence Change (A right to B wrong)": (
            confidence_change_correct_to_incorrect(dfA, dfB)
        ),
        "Confidence Change (A wrong to B right)": (
            confidence_change_incorrect_to_correct(dfA, dfB)
        ),
    })

pair_results_df = pd.DataFrame(pair_results).round(3)

pair_results_df

,Pair,Accuracy A,Accuracy B,Accuracy Gap (A - B),Mean Confidence A,Mean Confidence B,Mean Confidence Change (A to B),Mean Abs Confidence Diff,Items with Conf Diff >= 5,Answer Flips,Flip Rate,"A Right, B Wrong","B Right, A Wrong",McNemar p,Confidence Change (A right to B wrong),Confidence Change (A wrong to B right)
0,Baseline vs Misleading,0.417,0.183,0.233,88.167,90.833,2.667,3.667,19,38,0.633,17,3,0.003,4.118,3.333
1,Baseline vs Black,0.417,0.350,0.067,88.167,88.333,0.167,0.500,3,4,0.067,4,0,0.125,-2.500,NaN
2,Baseline vs White,0.417,0.350,0.067,88.167,88.667,0.500,0.500,3,4,0.067,4,0,0.125,2.500,NaN
3,Baseline vs Uninsurance,0.417,0.400,0.017,88.167,87.500,-0.667,1.000,6,4,0.067,2,1,1.000,-5.000,0.000
4,Baseline vs Insurance,0.417,0.350,0.067,88.167,88.000,-0.167,0.833,5,7,0.117,5,1,0.219,-2.000,0.000
5,Baseline vs Rural,0.417,0.367,0.050,88.167,88.000,-0.167,0.500,3,3,0.050,3,0,0.250,0.000,NaN
6,Baseline vs Urban,0.417,0.350,0.067,88.167,88.667,0.500,0.500,3,4,0.067,4,0,0.125,2.500,NaN
7,Baseline vs Injection,0.417,0.300,0.117,88.167,100.000,11.833,11.833,56,17,0.283,8,1,0.039,13.750,10.000
8,Baseline vs Canary,0.417,0.383,0.033,88.167,89.417,1.250,1.417,12,10,0.167,5,3,0.727,2.000,1.667


In [5]:

import pandas as pd
from scipy.stats import mannwhitneyu

conditions = {
    "Misleading context": df_5_1,
    "Prompt injection": df_5_4_1,
}

results = []

for name, df_condition in conditions.items():

    # Align baseline and condition results by item ID
    merged = df_4[
        ["id", "answer", "parsed_answer", "confidence"]
    ].merge(
        df_condition[
            ["id", "parsed_answer", "confidence"]
        ],
        on="id",
        suffixes=("_base", "_condition")
    )

    # Keep items answered correctly at baseline
    baseline_correct = (
        merged["parsed_answer_base"] == merged["answer"]
    )
    subset = merged[baseline_correct].copy()

    # Did the model fail after the intervention?
    subset["failed"] = (
        subset["parsed_answer_condition"] != subset["answer"]
    )

    failures = subset.loc[
        subset["failed"], "confidence_base"
    ]
    stable = subset.loc[
        ~subset["failed"], "confidence_base"
    ]

    # Compare baseline confidence between the two groups
    if len(failures) > 0 and len(stable) > 0:
        test = mannwhitneyu(
            failures, stable, alternative="two-sided"
        )
        p_value = test.pvalue
        u_statistic = test.statistic
    else:
        p_value = float("nan")
        u_statistic = float("nan")

    results.append({
        "Condition": name,
        "Baseline-correct items": len(subset),
        "Items that became incorrect": len(failures),
        "Items that stayed correct": len(stable),
        "Median baseline confidence (failures)": failures.median(),
        "Median baseline confidence (stable)": stable.median(),
        "Mean baseline confidence (failures)": failures.mean(),
        "Mean baseline confidence (stable)": stable.mean(),
        "Mann-Whitney U": u_statistic,
        "p-value": p_value,
    })

cross_dimension_results = pd.DataFrame(results).round(3)
cross_dimension_results


,Condition,Baseline-correct items,Items that became incorrect,Items that stayed correct,Median baseline confidence (failures),Median baseline confidence (stable),Mean baseline confidence (failures),Mean baseline confidence (stable),Mann-Whitney U,p-value
0,Misleading context,25,17,8,90.0,90.0,87.059,86.250,73.5,0.719
1,Prompt injection,25,8,17,90.0,90.0,86.250,87.059,62.5,0.719


In [6]:
import pandas as pd

df_4 = pd.read_csv(f'{base_dir}/qwen_csv/4_baseline_qwen_results.csv')
df_4 = df_4.head(60)
df_5_1 = pd.read_csv(f'{base_dir}/qwen_csv/5.1_hallicunation_and_uncertainity_qwen_results.csv')
df_5_3_1 = pd.read_csv(f'{base_dir}/qwen_csv/5.3_bias_and_fairness_race_black_qwen_results.csv')
df_5_3_2 = pd.read_csv(f'{base_dir}/qwen_csv/5.3_bias_and_fairness_race_white_qwen_results.csv')
df_5_3_3 = pd.read_csv(f'{base_dir}/qwen_csv/5.3_bias_and_fairness_insurance_uninsured_qwen_results.csv')
df_5_3_4 = pd.read_csv(f'{base_dir}/qwen_csv/5.3_bias_and_fairness_insurance_private_insurance_qwen_results.csv')
df_5_3_5 = pd.read_csv(f'{base_dir}/qwen_csv/5.3_bias_and_fairness_residence_rural_qwen_results.csv')
df_5_3_6 = pd.read_csv(f'{base_dir}/qwen_csv/5.3_bias_and_fairness_residence_urban_qwen_results.csv')
df_5_4_1 = pd.read_csv(f'{base_dir}/qwen_csv/5.4_prompt_injection_qwen_results.csv')
df_5_4_2 = pd.read_csv(f'{base_dir}/qwen_csv/5.4_canary_leakage_qwen_results.csv')

In [7]:
import pandas as pd
from scipy.stats import binomtest

pairs = {
    "Baseline vs Misleading": (df_4, df_5_1),
    "Baseline vs Black": (df_4, df_5_3_1),
    "Baseline vs White": (df_4, df_5_3_2),
    "Baseline vs Uninsurance": (df_4, df_5_3_3),
    "Baseline vs Insurance": (df_4, df_5_3_4),
    "Baseline vs Rural": (df_4, df_5_3_5),
    "Baseline vs Urban": (df_4, df_5_3_6),
    "Baseline vs Injection": (df_4, df_5_4_1),
    "Baseline vs Canary": (df_4, df_5_4_2),
}

pair_results = []

for name, (dfA, dfB) in pairs.items():
    merged = dfA.merge(dfB, on="id", suffixes=("_A", "_B"))

    conf_diff = merged["confidence_A"] - merged["confidence_B"]

    a_right_b_wrong = answer_flip_correct_to_incorrect(dfA, dfB)
    b_right_a_wrong = answer_flip_incorrect_to_correct(dfA, dfB)
    n_discordant = a_right_b_wrong + b_right_a_wrong

    pair_results.append({
        "Pair": name,
        "Accuracy A": accuracy(dfA),
        "Accuracy B": accuracy(dfB),
        "Accuracy Gap (A - B)": accuracy(dfA) - accuracy(dfB),
        "Mean Confidence A": mean_confidence_overall(dfA),
        "Mean Confidence B": mean_confidence_overall(dfB),
        "Mean Confidence Change (A to B)": mean_confidence_change(dfA, dfB),
        "Mean Abs Confidence Diff": conf_diff.abs().mean(),
        "Items with Conf Diff >= 5": (conf_diff.abs() >= 5).sum(),
        "Answer Flips": answer_flip_overall(dfA, dfB),
        "Flip Rate": answer_flip_overall(dfA, dfB) / len(merged),
        "A Right, B Wrong": a_right_b_wrong,
        "B Right, A Wrong": b_right_a_wrong,
        "McNemar p": (
            binomtest(
                min(a_right_b_wrong, b_right_a_wrong),
                n_discordant,
                0.5
            ).pvalue if n_discordant > 0 else 1.0
        ),
        "Confidence Change (A right to B wrong)": (
            confidence_change_correct_to_incorrect(dfA, dfB)
        ),
        "Confidence Change (A wrong to B right)": (
            confidence_change_incorrect_to_correct(dfA, dfB)
        ),
    })

pair_results_df = pd.DataFrame(pair_results).round(3)

pair_results_df

,Pair,Accuracy A,Accuracy B,Accuracy Gap (A - B),Mean Confidence A,Mean Confidence B,Mean Confidence Change (A to B),Mean Abs Confidence Diff,Items with Conf Diff >= 5,Answer Flips,Flip Rate,"A Right, B Wrong","B Right, A Wrong",McNemar p,Confidence Change (A right to B wrong),Confidence Change (A wrong to B right)
0,Baseline vs Misleading,0.333,0.133,0.200,94.083,95.000,0.917,1.583,17,44,0.733,15,3,0.008,1.333,-1.667
1,Baseline vs Black,0.333,0.300,0.033,94.083,93.500,-0.583,0.583,7,4,0.067,3,1,0.625,0.000,0.000
2,Baseline vs White,0.333,0.300,0.033,94.083,93.500,-0.583,0.583,7,6,0.100,3,1,0.625,0.000,0.000
3,Baseline vs Uninsurance,0.333,0.333,0.000,94.083,93.333,-0.750,0.750,9,7,0.117,3,3,1.000,0.000,-3.333
4,Baseline vs Insurance,0.333,0.333,0.000,94.083,93.417,-0.667,0.667,8,8,0.133,3,3,1.000,0.000,-3.333
5,Baseline vs Rural,0.333,0.333,0.000,94.083,93.500,-0.583,0.583,7,2,0.033,1,1,1.000,0.000,0.000
6,Baseline vs Urban,0.333,0.317,0.017,94.083,93.500,-0.583,0.583,7,3,0.050,2,1,1.000,0.000,0.000
7,Baseline vs Injection,0.333,0.133,0.200,94.083,99.917,5.833,5.833,57,37,0.617,14,2,0.004,6.071,7.500
8,Baseline vs Canary,0.333,0.300,0.033,94.083,95.083,1.000,1.500,17,10,0.167,5,3,0.727,3.000,0.000


In [8]:
import pandas as pd
from scipy.stats import mannwhitneyu

conditions = {
    "Misleading context": df_5_1,
    "Prompt injection": df_5_4_1,
}

results = []

for name, df_condition in conditions.items():

    # Align baseline and condition results by item ID
    merged = df_4[
        ["id", "answer", "parsed_answer", "confidence"]
    ].merge(
        df_condition[
            ["id", "parsed_answer", "confidence"]
        ],
        on="id",
        suffixes=("_base", "_condition")
    )

    # Keep items answered correctly at baseline
    baseline_correct = (
        merged["parsed_answer_base"] == merged["answer"]
    )
    subset = merged[baseline_correct].copy()

    # Did the model fail after the intervention?
    subset["failed"] = (
        subset["parsed_answer_condition"] != subset["answer"]
    )

    failures = subset.loc[
        subset["failed"], "confidence_base"
    ]
    stable = subset.loc[
        ~subset["failed"], "confidence_base"
    ]

    # Compare baseline confidence between the two groups
    if len(failures) > 0 and len(stable) > 0:
        test = mannwhitneyu(
            failures, stable, alternative="two-sided"
        )
        p_value = test.pvalue
        u_statistic = test.statistic
    else:
        p_value = float("nan")
        u_statistic = float("nan")

    results.append({
        "Condition": name,
        "Baseline-correct items": len(subset),
        "Items that became incorrect": len(failures),
        "Items that stayed correct": len(stable),
        "Median baseline confidence (failures)": failures.median(),
        "Median baseline confidence (stable)": stable.median(),
        "Mean baseline confidence (failures)": failures.mean(),
        "Mean baseline confidence (stable)": stable.mean(),
        "Mann-Whitney U": u_statistic,
        "p-value": p_value,
    })

cross_dimension_results = pd.DataFrame(results).round(3)
cross_dimension_results


,Condition,Baseline-correct items,Items that became incorrect,Items that stayed correct,Median baseline confidence (failures),Median baseline confidence (stable),Mean baseline confidence (failures),Mean baseline confidence (stable),Mann-Whitney U,p-value
0,Misleading context,20,15,5,95.0,95.0,94.000,95.0,30.0,0.324
1,Prompt injection,20,14,6,95.0,95.0,93.929,95.0,33.0,0.258
